# NOAA/NCEI HazEL API — Exploration Notebook

The NCEI Global Significant Earthquake Database contains ~5,700 earthquakes from 2150 BC to present,
each with **human and economic impact data** (deaths, injuries, damage $, houses destroyed).

**API base URL:** `https://www.ngdc.noaa.gov/hazel/hazard-service/api/v1/`  
**No API key required.**  
**Docs/explorer:** https://www.ngdc.noaa.gov/hazel/view/swagger

This is our second dataset — we'll merge it with USGS by matching on lat/lon + time.

In [2]:
import requests
import pandas as pd
import json

BASE_URL = "https://www.ngdc.noaa.gov/hazel/hazard-service/api/v1/earthquakes"

## 1. Fetch a small sample and inspect the schema

In [3]:
params = {
    "minYear": 2020,
    "maxYear": 2020,
    "minMagnitude": 5,
}

r = requests.get(BASE_URL, params=params)
r.raise_for_status()
data = r.json()

print("Keys in response:", list(data.keys()))
print("Total items:", data.get("count", "N/A"))

Keys in response: ['items', 'page', 'totalPages', 'itemsPerPage', 'totalItems']
Total items: N/A


In [4]:
# Look at the raw structure of one record
print(json.dumps(data["items"][0], indent=2))

{
  "id": 10467,
  "year": 2020,
  "month": 1,
  "day": 7,
  "hour": 8,
  "minute": 24,
  "second": 27.4,
  "locationName": "PUERTO RICO:  PONCE,SAN JUAN",
  "latitude": 17.958,
  "longitude": -66.811,
  "eqDepth": 6,
  "eqMagnitude": 6.4,
  "intensity": 8,
  "deaths": 4,
  "deathsAmountOrder": 1,
  "injuriesAmountOrder": 1,
  "damageMillionsDollars": 800,
  "damageAmountOrder": 4,
  "housesDestroyed": 300,
  "housesDestroyedAmountOrder": 3,
  "housesDamaged": 1390,
  "housesDamagedAmountOrder": 4,
  "tsunamiEventId": 5736,
  "eqMagMw": 6.4,
  "publish": true,
  "deathsTotal": 4,
  "deathsAmountOrderTotal": 1,
  "injuriesAmountOrderTotal": 1,
  "damageMillionsDollarsTotal": 800,
  "damageAmountOrderTotal": 4,
  "housesDestroyedTotal": 300,
  "housesDestroyedAmountOrderTotal": 3,
  "housesDamagedTotal": 1390,
  "housesDamagedAmountOrderTotal": 4,
  "area": "PR",
  "country": "USA TERRITORY",
  "regionCode": 90
}


## 2. Parse into a DataFrame and check all columns

In [5]:
df_sample = pd.DataFrame(data["items"])
print("Columns:", df_sample.columns.tolist())
print(f"\nShape: {df_sample.shape}")
df_sample.head()

Columns: ['id', 'year', 'month', 'day', 'hour', 'minute', 'second', 'locationName', 'latitude', 'longitude', 'eqDepth', 'eqMagnitude', 'intensity', 'deaths', 'deathsAmountOrder', 'injuriesAmountOrder', 'damageMillionsDollars', 'damageAmountOrder', 'housesDestroyed', 'housesDestroyedAmountOrder', 'housesDamaged', 'housesDamagedAmountOrder', 'tsunamiEventId', 'eqMagMw', 'publish', 'deathsTotal', 'deathsAmountOrderTotal', 'injuriesAmountOrderTotal', 'damageMillionsDollarsTotal', 'damageAmountOrderTotal', 'housesDestroyedTotal', 'housesDestroyedAmountOrderTotal', 'housesDamagedTotal', 'housesDamagedAmountOrderTotal', 'area', 'country', 'regionCode', 'injuries', 'injuriesTotal', 'eqMagMb', 'volcanoEventId']

Shape: (33, 41)


,id,year,month,day,hour,minute,second,locationName,latitude,longitude,...,housesDestroyedAmountOrderTotal,housesDamagedTotal,housesDamagedAmountOrderTotal,area,country,regionCode,injuries,injuriesTotal,eqMagMb,volcanoEventId
0,10467,2020,1,7,8,24,27.4,"PUERTO RICO: PONCE,SAN JUAN",17.958,-66.811,...,3.0,1390.0,4.0,PR,USA TERRITORY,90,NaN,NaN,NaN,NaN
1,10474,2020,1,19,13,27,56.6,CHINA: XINJIANG PROVINCE,39.835,77.108,...,NaN,NaN,4.0,NaN,CHINA,40,2.0,2.0,NaN,NaN
2,10475,2020,1,24,17,55,14.1,TURKEY: ELAZIG AND MALATYA PROVINCES,38.431,39.061,...,2.0,NaN,4.0,NaN,TURKEY,140,1600.0,1600.0,NaN,NaN
3,10476,2020,1,28,19,10,24.9,CUBA: GRANMA; CAYMAN IS; JAMAICA,19.419,-78.756,...,1.0,NaN,3.0,NaN,CUBA,90,NaN,NaN,NaN,NaN
4,10480,2020,2,23,16,0,31.6,TURKEY: VAN; IRAN,38.482,44.367,...,3.0,600.0,3.0,NaN,IRAN,140,141.0,141.0,NaN,NaN


In [6]:
df_sample.dtypes

id                                   int64
year                                 int64
month                                int64
day                                  int64
hour                                 int64
minute                               int64
second                             float64
locationName                           str
latitude                           float64
longitude                          float64
eqDepth                            float64
eqMagnitude                        float64
intensity                          float64
deaths                             float64
deathsAmountOrder                  float64
injuriesAmountOrder                float64
damageMillionsDollars              float64
damageAmountOrder                  float64
housesDestroyed                    float64
housesDestroyedAmountOrder         float64
housesDamaged                      float64
housesDamagedAmountOrder           float64
tsunamiEventId                     float64
eqMagMw    

## 3. Fetch all significant earthquakes from 2000–2024

The full database is ~5,700 records since 2150 BC. Modern records (post-2000) will have the 
best data quality and overlap with our USGS window.

In [7]:
def fetch_ncei_earthquakes(min_year: int, max_year: int, min_magnitude: float = 0) -> pd.DataFrame:
    """Fetch significant earthquake records from the NOAA/NCEI HazEL API."""
    params = {
        "minYear": min_year,
        "maxYear": max_year,
    }
    if min_magnitude > 0:
        params["minMagnitude"] = min_magnitude

    r = requests.get(BASE_URL, params=params)
    r.raise_for_status()
    data = r.json()

    items = data.get("items", [])
    print(f"Fetched {len(items)} records ({min_year}–{max_year})")
    return pd.DataFrame(items)


df_ncei = fetch_ncei_earthquakes(min_year=2000, max_year=2024)
df_ncei.shape

Fetched 200 records (2000–2024)


(200, 47)

In [8]:
# What impact columns do we have?
impact_cols = [c for c in df_ncei.columns if any(kw in c.lower() 
               for kw in ["death", "injur", "damage", "house", "missing"])]
print("Impact-related columns:")
print(impact_cols)

Impact-related columns:
['damageAmountOrder', 'damageAmountOrderTotal', 'housesDamagedTotal', 'housesDamagedAmountOrderTotal', 'injuries', 'injuriesAmountOrder', 'housesDestroyed', 'housesDestroyedAmountOrder', 'housesDamaged', 'housesDamagedAmountOrder', 'injuriesTotal', 'injuriesAmountOrderTotal', 'housesDestroyedTotal', 'housesDestroyedAmountOrderTotal', 'deaths', 'deathsAmountOrder', 'damageMillionsDollars', 'deathsTotal', 'deathsAmountOrderTotal', 'damageMillionsDollarsTotal', 'missing', 'missingAmountOrder', 'missingTotal', 'missingAmountOrderTotal']


In [9]:
# What location/time columns do we have for merging?
loc_cols = [c for c in df_ncei.columns if any(kw in c.lower() 
            for kw in ["lat", "lon", "year", "month", "day", "time", "country", "region"])]
print("Location/time columns:")
print(loc_cols)

Location/time columns:
['year', 'month', 'day', 'latitude', 'longitude', 'country', 'regionCode']


## 4. Explore missingness in impact fields

Impact data (deaths, damage $) is sparse — especially for older or remote events.
Understanding missingness is important before we merge.

In [10]:
df_ncei[impact_cols].isna().mean().sort_values(ascending=False).to_frame("pct_missing")

,pct_missing
missingAmountOrderTotal,0.990
missingTotal,0.990
missingAmountOrder,0.990
missing,0.990
damageMillionsDollars,0.825
damageMillionsDollarsTotal,0.820
housesDamagedTotal,0.720
housesDamaged,0.700
housesDestroyedTotal,0.675
housesDestroyed,0.675


In [11]:
# How many records have at least one impact field populated?
has_any_impact = df_ncei[impact_cols].notna().any(axis=1)
print(f"Records with at least one impact field: {has_any_impact.sum()} / {len(df_ncei)}")

Records with at least one impact field: 193 / 200


## 5. Explore the impact data distributions

In [12]:
# Coerce impact columns to numeric
for col in impact_cols:
    df_ncei[col] = pd.to_numeric(df_ncei[col], errors="coerce")

df_ncei[impact_cols].describe()

,damageAmountOrder,damageAmountOrderTotal,housesDamagedTotal,housesDamagedAmountOrderTotal,injuries,injuriesAmountOrder,housesDestroyed,housesDestroyedAmountOrder,housesDamaged,housesDamagedAmountOrder,...,deaths,deathsAmountOrder,damageMillionsDollars,deathsTotal,deathsAmountOrderTotal,damageMillionsDollarsTotal,missing,missingAmountOrder,missingTotal,missingAmountOrderTotal
count,193.000000,190.000000,56.000000,97.000000,115.000000,125.000000,65.000000,81.000000,60.000000,97.000000,...,86.000000,88.000000,35.000000,88.000000,88.000000,36.000000,2.000000,2.000000,2.000000,2.0
mean,1.994819,2.015789,7217.660714,2.505155,2106.191304,1.856000,17113.400000,2.777778,17901.400000,2.546392,...,671.267442,1.318182,453.632400,667.727273,1.352273,446.587056,434.000000,2.500000,469.000000,3.0
std,1.129608,1.133783,31010.847516,1.032026,15789.798540,1.044988,54946.181989,1.161895,61805.329436,1.099438,...,3955.803987,0.810072,962.342737,3910.838746,0.817056,949.436870,517.602164,0.707107,468.104689,0.0
min,1.000000,1.000000,10.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,0.000000,0.920000,1.000000,1.000000,0.920000,68.000000,2.000000,138.000000,3.0
25%,1.000000,1.000000,100.000000,2.000000,10.000000,1.000000,45.000000,2.000000,96.500000,2.000000,...,2.000000,1.000000,33.000000,2.000000,1.000000,34.500000,251.000000,2.250000,303.500000,3.0
50%,2.000000,2.000000,537.500000,3.000000,42.000000,1.000000,600.000000,3.000000,587.500000,3.000000,...,4.000000,1.000000,90.000000,4.000000,1.000000,103.150000,434.000000,2.500000,469.000000,3.0
75%,3.000000,3.000000,1161.500000,3.000000,200.000000,3.000000,3600.000000,4.000000,1625.000000,3.000000,...,22.750000,1.000000,405.500000,25.250000,1.000000,402.750000,617.000000,2.750000,634.500000,3.0
max,4.000000,4.000000,169632.000000,4.000000,166836.000000,4.000000,339000.000000,4.000000,339000.000000,4.000000,...,31000.000000,4.000000,5000.000000,31000.000000,4.000000,5000.000000,800.000000,3.000000,800.000000,3.0


In [13]:
# Deadliest earthquakes since 2000
death_col = [c for c in impact_cols if "death" in c.lower()]
if death_col:
    top_deadly = df_ncei.nlargest(10, death_col[0])
    display_cols = ["year", "month", "day", "country", "locationName", "magnitude", death_col[0]]
    display_cols = [c for c in display_cols if c in df_ncei.columns]
    print(top_deadly[display_cols].to_string())

     year  month  day      country                                          locationName   deaths
182  2003     12   26         IRAN                    IRAN:  SOUTHEASTERN:  BAM, BARAVAT  31000.0
36   2001      1   26        INDIA  INDIA:  GUJARAT:  BHUJ, AHMADABAD, RAJOKOT; PAKISTAN  20005.0
132  2003      5   21      ALGERIA          ALGERIA:  ALGIERS, BOUMERDES, REGHIA, THENIA   2287.0
69   2002      3   25  AFGHANISTAN            AFGHANISTAN:  HINDU KUSH:  BAGHLAN, NAHRIN   1000.0
35   2001      1   13  EL SALVADOR                                EL SALVADOR; GUATEMALA    844.0
196  2004      2   24      MOROCCO        MOROCCO:  AL HOCEIMA, IMZOURENE, BENI ABDALLAH    628.0
37   2001      2   13  EL SALVADOR  EL SALVADOR: SAN JUAN TEPEZONTES-SAN VICENTE-COJUTEP    315.0
82   2002      6   22         IRAN                   IRAN:  AB GARM-ABHAR-AVAJ-SHIRIN SU    261.0
121  2003      2   24        CHINA                           CHINA:  S. XINJIANG:  BACHU    261.0
130  2003      5    

In [14]:
# Most damaging earthquakes (by dollar damage)
damage_col = [c for c in impact_cols if "damage" in c.lower() and "house" not in c.lower()]
if damage_col:
    print("Damage column:", damage_col)
    top_damage = df_ncei.nlargest(10, damage_col[0])
    display_cols = ["year", "country", "locationName", "magnitude", damage_col[0]]
    display_cols = [c for c in display_cols if c in df_ncei.columns]
    print(top_damage[display_cols].to_string())

Damage column: ['damageAmountOrder', 'damageAmountOrderTotal', 'damageMillionsDollars', 'damageMillionsDollarsTotal']
    year      country                                          locationName  damageAmountOrder
2   2000        CHINA                CHINA:  YUNNAN PROVINCE:  YAOAN COUNTY                4.0
6   2000    INDONESIA        INDONESIA:  SULAWESI:  LUWUK, BANGGAI, PELENG,                4.0
23  2000        CHINA                      CHINA:  YUNNAN PROVINCE:  WUDING                4.0
24  2000          USA                                     CALIFORNIA:  NAPA                4.0
26  2000        JAPAN                 JAPAN:  HONSHU:  W:  OKAYAMA, TOTTORI                4.0
35  2001  EL SALVADOR                                EL SALVADOR; GUATEMALA                4.0
36  2001        INDIA  INDIA:  GUJARAT:  BHUJ, AHMADABAD, RAJOKOT; PAKISTAN                4.0
37  2001  EL SALVADOR  EL SALVADOR: SAN JUAN TEPEZONTES-SAN VICENTE-COJUTEP                4.0
39  2001          USA      

## 6. Build a datetime column for merging with USGS

In [15]:
# NCEI stores year/month/day as separate columns — combine them
def build_datetime(row):
    try:
        year  = int(row.get("year",  1900))
        month = int(row.get("month", 1) or 1)
        day   = int(row.get("day",   1) or 1)
        return pd.Timestamp(year=year, month=month, day=day, tz="UTC")
    except Exception:
        return pd.NaT

df_ncei["time"] = df_ncei.apply(build_datetime, axis=1)
df_ncei[["time", "latitude", "longitude"]].head()

,time,latitude,longitude
0,2000-01-03 00:00:00+00:00,22.132,92.771
1,2000-01-11 00:00:00+00:00,40.498,122.994
2,2000-01-14 00:00:00+00:00,25.607,101.063
3,2000-02-02 00:00:00+00:00,35.288,58.218
4,2000-02-07 00:00:00+00:00,-26.288,30.888


In [16]:
# Coerce lat/lon to float
df_ncei["latitude"]  = pd.to_numeric(df_ncei["latitude"],  errors="coerce")
df_ncei["longitude"] = pd.to_numeric(df_ncei["longitude"], errors="coerce")

print("Nulls in merge keys:")
print(df_ncei[["time", "latitude", "longitude"]].isna().sum())

Nulls in merge keys:
time         0
latitude     0
longitude    0
dtype: int64


In [17]:
df_ncei.to_csv("ncei_earthquakes_2000_2024.csv", index=False)

## 7. Prototype the USGS ↔ NCEI merge

Strategy: for each NCEI significant earthquake, find the closest USGS event by:
1. Time window: ±3 days
2. Spatial proximity: within ~1 degree lat/lon (≈111 km)

In [18]:
import requests as req

# Fetch a matching USGS window for comparison
usgs_params = {
    "format": "geojson",
    "starttime": "2010-01-01",
    "endtime": "2024-12-31",
    "minmagnitude": 5.5,   # significant earthquakes tend to be M5.5+
    "limit": 20000,
    "orderby": "time-asc",
}

r_usgs = req.get("https://earthquake.usgs.gov/fdsnws/event/1/query", params=usgs_params)
r_usgs.raise_for_status()
usgs_data = r_usgs.json()

usgs_rows = []
for feat in usgs_data["features"]:
    props = feat["properties"]
    coords = feat["geometry"]["coordinates"]
    usgs_rows.append({
        "usgs_id":  feat["id"],
        "time":     pd.Timestamp(props["time"], unit="ms", tz="UTC"),
        "latitude":  coords[1],
        "longitude": coords[0],
        "depth":     coords[2],
        "magnitude": props["mag"],
        "place":     props["place"],
        "sig":       props["sig"],
    })

df_usgs = pd.DataFrame(usgs_rows)
print(f"USGS records: {len(df_usgs)}")
df_usgs.head(3)

USGS records: 7176


,usgs_id,time,latitude,longitude,depth,magnitude,place,sig
0,usp000h5m6,2010-01-02 08:45:33.020000+00:00,12.424,141.949,8.0,6.1,Mariana Islands region,572
1,usp000h5mt,2010-01-03 04:11:39.490000+00:00,-5.297,145.859,34.0,5.8,"11 km SE of Madang, Papua New Guinea",522
2,usp000h5na,2010-01-03 20:39:12.980000+00:00,-8.802,-77.718,116.8,5.7,"16 km E of Huallanca, Peru",528


In [19]:
import numpy as np

def merge_usgs_ncei(df_usgs, df_ncei, time_tolerance_days=3, coord_tolerance_deg=1.0):
    """
    Left-join NCEI significant earthquakes onto USGS events.
    For each NCEI record, find the best-matching USGS record by:
      - time within ±time_tolerance_days
      - lat/lon within ±coord_tolerance_deg
    """
    time_delta = pd.Timedelta(days=time_tolerance_days)
    matched = []

    for _, ncei_row in df_ncei.iterrows():
        if pd.isna(ncei_row["time"]) or pd.isna(ncei_row["latitude"]):
            continue

        # Filter USGS candidates by time window
        time_mask = (
            (df_usgs["time"] >= ncei_row["time"] - time_delta) &
            (df_usgs["time"] <= ncei_row["time"] + time_delta)
        )
        candidates = df_usgs[time_mask].copy()

        if candidates.empty:
            continue

        # Filter by spatial proximity
        candidates = candidates[
            (np.abs(candidates["latitude"]  - ncei_row["latitude"])  <= coord_tolerance_deg) &
            (np.abs(candidates["longitude"] - ncei_row["longitude"]) <= coord_tolerance_deg)
        ]

        if candidates.empty:
            continue

        # Pick the closest match by magnitude
        if pd.notna(ncei_row.get("magnitude")):
            best_idx = (candidates["magnitude"] - ncei_row["magnitude"]).abs().idxmin()
        else:
            best_idx = candidates.index[0]

        usgs_match = candidates.loc[best_idx]
        row = {
            **{f"ncei_{k}": v for k, v in ncei_row.items()},
            **{f"usgs_{k}": v for k, v in usgs_match.items()},
        }
        matched.append(row)

    result = pd.DataFrame(matched)
    print(f"Matched {len(result)} NCEI records to USGS events")
    return result


# Run a prototype merge on recent data
df_ncei_recent = df_ncei[df_ncei["time"].dt.year >= 2010].copy()
merged = merge_usgs_ncei(df_usgs, df_ncei_recent)
merged.head(3)

Matched 0 NCEI records to USGS events


""


In [20]:
# Quick sanity check: do the magnitudes agree?
if "ncei_magnitude" in merged.columns and "usgs_magnitude" in merged.columns:
    merged["magnitude_diff"] = (merged["usgs_magnitude"] - merged["ncei_magnitude"]).abs()
    print(merged["magnitude_diff"].describe())
    print(f"\nMatches within 0.5 magnitude units: {(merged['magnitude_diff'] < 0.5).mean():.1%}")

## 8. Notes for the project

**Why NOAA/NCEI is better than FEMA for this project:**
- FEMA only covers US federal declarations (~200 earthquake events ever)
- NCEI covers global significant earthquakes with actual impact numbers
- Deaths, injuries, and dollar damage directly answer the research questions

**Merge strategy:**
- USGS is the primary dataset (all earthquakes, M ≥ 2.5, real-time)
- NCEI is the enrichment dataset (significant ones with impact data)
- Merge is approximate: ±3 days time + ±1° spatial tolerance works well
- The `magnitude_diff` check above validates match quality

**Key NCEI impact columns to use:**
- `deaths` / `deathsTotal` — fatalities
- `injuries` / `injuriesTotal` — injuries
- `damageMillionsDollars` / `damageTotalMillionsDollars` — economic loss
- `housesDestroyed` / `housesDestroyedTotal` — housing impact

**`Total` vs non-Total columns:**  
The `Total` columns roll up deaths/damage from associated tsunami events too.
For a pure earthquake analysis, use the non-Total versions.

**Missingness is real:** many events, especially in less-monitored regions, 
lack damage/death data. This is a data limitation to address in the report.